# 고성능 Pandas: 평가 및 쿼리

이전 장에서 이미 살펴본 것처럼 PyData 스택의 강력한 기능은 직관적인 상위 수준 구문을 통해 기본 작업을 하위 수준의 컴파일된 코드로 푸시하는 NumPy 및 Pandas의 기능을 기반으로 구축되었습니다. 예는 NumPy의 벡터화/브로드캐스트 작업과 Pandas의 그룹화 유형 작업입니다.
이러한 추상화는 많은 일반적인 사용 사례에서 효율적이고 효과적이지만 임시 중간 개체 생성에 의존하는 경우가 많으며 이로 인해 계산 시간과 메모리 사용에 과도한 오버헤드가 발생할 수 있습니다.

이 문제를 해결하기 위해 Pandas에는 비용이 많이 드는 중간 배열을 할당하지 않고도 C 속도 작업에 직접 액세스할 수 있는 몇 가지 메서드인 [NumExpr 패키지](https://github.com/pydata/numexpr)를 사용하는 `eval` 및 `query`가 포함되어 있습니다.
이번 장에서는 사용법을 안내하고 언제 사용을 고려할 수 있는지에 대한 몇 가지 경험 법칙을 제시하겠습니다.

## 쿼리 및 평가 동기 부여: 복합 표현식

우리는 이전에 NumPy와 Pandas가 빠른 벡터화 작업을 지원한다는 것을 확인했습니다. 예를 들어 두 배열의 요소를 추가하는 경우:

In [1]:
import numpy as np
rng = np.random.default_rng(42)
x = rng.random(1000000)
y = rng.random(1000000)
%timeit x + y

2.21 ms ± 142 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


[NumPy 배열 계산: 범용 함수](02.03-Computation-on-arrays-ufuncs.ipynb)에서 설명한 대로 이는 파이썬(Python) 루프나 이해를 통해 추가하는 것보다 훨씬 빠릅니다.

In [2]:
%timeit np.fromiter((xi + yi for xi, yi in zip(x, y)),
                    dtype=x.dtype, count=len(x))

263 ms ± 43.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


그러나 이 추상화는 복합 표현식을 계산할 때 효율성이 떨어질 수 있습니다.
예를 들어 다음 표현식을 고려해보세요.

In [3]:
mask = (x > 0.5) & (y < 0.5)

NumPy는 각 하위 표현식을 평가하므로 이는 대략 다음과 동일합니다.

In [4]:
tmp1 = (x > 0.5)
tmp2 = (y < 0.5)
mask = tmp1 & tmp2

즉, *모든 중간 단계는 명시적으로 메모리에 할당됩니다*. `x` 및 `y` 배열이 매우 큰 경우 상당한 메모리와 계산 오버헤드가 발생할 수 있습니다.
NumExpr 라이브러리는 전체 중간 배열을 할당할 필요 없이 이러한 유형의 복합 표현식 요소를 요소별로 계산할 수 있는 기능을 제공합니다.
[NumExpr 문서](https://github.com/pydata/numexpr)에 더 자세한 내용이 있지만 당분간은 라이브러리가 계산하려는 NumPy 스타일 표현식을 제공하는 *문자열*을 허용한다고 말하는 것으로 충분합니다.

In [5]:
import numexpr
mask_numexpr = numexpr.evaluate('(x > 0.5) & (y < 0.5)')
np.all(mask == mask_numexpr)

True

여기서의 이점은 NumExpr이 가능한 경우 임시 배열을 피하는 방식으로 표현식을 평가하므로 NumPy보다 훨씬 더 효율적일 수 있다는 것입니다. 특히 대규모 배열에 대한 긴 계산 시퀀스의 경우 더욱 그렇습니다.
여기서 논의할 Pandas 'eval' 및 'query' 도구는 개념적으로 유사하며 본질적으로 NumExpr 기능의 Pandas 전용 래퍼입니다.

효율적인 운영을 위한 ## pandas.eval

Pandas의 `eval` 함수는 문자열 표현식을 사용하여 `DataFrame` 객체에 대한 작업을 효율적으로 계산합니다.
예를 들어 다음 데이터를 고려해보세요.

In [6]:
import pandas as pd
nrows, ncols = 100000, 100
df1, df2, df3, df4 = (pd.DataFrame(rng.random((nrows, ncols)))
                      for i in range(4))

일반적인 Pandas 접근 방식을 사용하여 4개의 ``DataFrame`` 모두의 합계를 계산하려면 다음과 같이 합계를 작성하면 됩니다.

In [7]:
%timeit df1 + df2 + df3 + df4

73.2 ms ± 6.72 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


표현식을 문자열로 구성하여 ``pd.eval``을 통해 동일한 결과를 계산할 수 있습니다.

In [8]:
%timeit pd.eval('df1 + df2 + df3 + df4')

34 ms ± 4.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


이 표현식의 'eval' 버전은 약 50% 더 빠르고(그리고 훨씬 적은 메모리를 사용하며) 동일한 결과를 제공합니다.

In [9]:
np.allclose(df1 + df2 + df3 + df4,
            pd.eval('df1 + df2 + df3 + df4'))

True

`pd.eval`은 광범위한 작업을 지원합니다.
이를 설명하기 위해 다음 정수 데이터를 사용합니다.

In [10]:
df1, df2, df3, df4, df5 = (pd.DataFrame(rng.integers(0, 1000, (100, 3)))
                           for i in range(5))

#### 산술 연산자
`pd.eval`은 모든 산술 연산자를 지원합니다. 예를 들어:

In [11]:
result1 = -df1 * df2 / (df3 + df4) - df5
result2 = pd.eval('-df1 * df2 / (df3 + df4) - df5')
np.allclose(result1, result2)

True

#### 비교 연산자
`pd.eval`은 연결된 표현식을 포함한 모든 비교 연산자를 지원합니다.

In [12]:
result1 = (df1 < df2) & (df2 <= df3) & (df3 != df4)
result2 = pd.eval('df1 < df2 <= df3 != df4')
np.allclose(result1, result2)

True

#### 비트 연산자
`pd.eval`은 `&` 및 `|` 비트 연산자를 지원합니다:

In [13]:
result1 = (df1 < 0.5) & (df2 < 0.5) | (df3 < df4)
result2 = pd.eval('(df1 < 0.5) & (df2 < 0.5) | (df3 < df4)')
np.allclose(result1, result2)

True

또한 부울 표현식에서 리터럴 `and` 및 `or` 사용을 지원합니다.

In [14]:
result3 = pd.eval('(df1 < 0.5) and (df2 < 0.5) or (df3 < df4)')
np.allclose(result1, result3)

True

#### 객체 속성 및 인덱스

`pd.eval`은 `obj.attr` 구문을 통해 객체 속성에 대한 액세스를 지원하고 `obj[index]` 구문을 통해 인덱스에 대한 액세스를 지원합니다.

In [15]:
result1 = df2.T[0] + df3.iloc[1]
result2 = pd.eval('df2.T[0] + df3.iloc[1]')
np.allclose(result1, result2)

True

#### 기타 작업

함수 호출, 조건문, 루프 및 기타 관련 구성과 같은 기타 작업은 현재 `pd.eval`에 구현되지 *않습니다*.
이러한 더 복잡한 유형의 표현식을 실행하려면 NumExpr 라이브러리 자체를 사용할 수 있습니다.

## 열 단위 작업을 위한 DataFrame.eval

Pandas에 최상위 `pd.eval` 함수가 있는 것처럼 `DataFrame` 객체에는 비슷한 방식으로 작동하는 `eval` 메서드가 있습니다.
'eval' 메서드의 장점은 열을 이름으로 참조할 수 있다는 것입니다.
이 레이블이 지정된 배열을 예로 사용하겠습니다.

In [16]:
df = pd.DataFrame(rng.random((1000, 3)), columns=['A', 'B', 'C'])
df.head()

,A,B,C
0,0.850888,0.966709,0.958690
1,0.820126,0.385686,0.061402
2,0.059729,0.831768,0.652259
3,0.244774,0.140322,0.041711
4,0.818205,0.753384,0.578851


이전 섹션에서와 같이 `pd.eval`을 사용하면 다음과 같이 세 개의 열이 있는 표현식을 계산할 수 있습니다.

In [17]:
result1 = (df['A'] + df['B']) / (df['C'] - 1)
result2 = pd.eval("(df.A + df.B) / (df.C - 1)")
np.allclose(result1, result2)

True

`DataFrame.eval` 메서드를 사용하면 열이 포함된 표현식을 훨씬 더 간결하게 평가할 수 있습니다.

In [18]:
result3 = df.eval('(A + B) / (C - 1)')
np.allclose(result1, result3)

True

여기서는 평가된 표현식 내에서 *열 이름을 변수*로 처리하고 그 결과는 우리가 원하는 대로라는 점에 유의하세요.

### DataFrame.eval의 할당

방금 설명한 옵션 외에도 `DataFrame.eval`을 사용하면 모든 열에 할당할 수 있습니다.
`'A'`, `'B'`, `'C'` 열이 있는 이전의 `DataFrame`을 사용해 보겠습니다.

In [19]:
df.head()

,A,B,C
0,0.850888,0.966709,0.958690
1,0.820126,0.385686,0.061402
2,0.059729,0.831768,0.652259
3,0.244774,0.140322,0.041711
4,0.818205,0.753384,0.578851


`df.eval`을 사용하여 새 열 `'D'`를 생성하고 다른 열에서 계산된 값을 여기에 할당할 수 있습니다.

In [20]:
df.eval('D = (A + B) / C', inplace=True)
df.head()

,A,B,C,D
0,0.850888,0.966709,0.958690,1.895916
1,0.820126,0.385686,0.061402,19.638139
2,0.059729,0.831768,0.652259,1.366782
3,0.244774,0.140322,0.041711,9.232370
4,0.818205,0.753384,0.578851,2.715013


같은 방법으로 기존 열을 수정할 수 있습니다.

In [21]:
df.eval('D = (A - B) / C', inplace=True)
df.head()

,A,B,C,D
0,0.850888,0.966709,0.958690,-0.120812
1,0.820126,0.385686,0.061402,7.075399
2,0.059729,0.831768,0.652259,-1.183638
3,0.244774,0.140322,0.041711,2.504142
4,0.818205,0.753384,0.578851,0.111982


### DataFrame.eval의 지역 변수

`DataFrame.eval` 메서드는 로컬 파이썬(Python) 변수와 함께 작동할 수 있는 추가 구문을 지원합니다.
다음을 고려하십시오.

In [22]:
column_mean = df.mean(1)
result1 = df['A'] + column_mean
result2 = df.eval('A + @column_mean')
np.allclose(result1, result2)

True

여기서 `@` 문자는 *열 이름*이 아닌 *변수 이름*을 표시하며 두 개의 "네임스페이스", 즉 열의 네임스페이스와 파이썬(Python) 객체의 네임스페이스와 관련된 표현식을 효율적으로 평가할 수 있게 해줍니다.
이 `@` 문자는 `pandas.eval` *함수*가 아닌 `DataFrame.eval` *메소드*에서만 지원됩니다. `pandas.eval` 함수는 one(파이썬(Python)) 네임스페이스에만 액세스할 수 있기 때문입니다.

## DataFrame.query 메서드

`DataFrame`에는 평가된 문자열을 기반으로 하는 `query`라는 또 다른 메서드가 있습니다.
다음을 고려하십시오.

In [23]:
result1 = df[(df.A < 0.5) & (df.B < 0.5)]
result2 = pd.eval('df[(df.A < 0.5) & (df.B < 0.5)]')
np.allclose(result1, result2)

True

`DataFrame.eval` 논의에 사용된 예와 마찬가지로 이는 `DataFrame`의 열과 ​​관련된 표현식입니다.
하지만 `DataFrame.eval` 구문으로는 표현할 수 없습니다!
대신 이러한 유형의 필터링 작업에는 `query` 메서드를 사용할 수 있습니다.

In [24]:
result2 = df.query('A < 0.5 and B < 0.5')
np.allclose(result1, result2)

True

더 효율적인 계산일 뿐만 아니라 마스킹 표현식에 비해 읽고 이해하기가 훨씬 쉽습니다.
`query` 메소드는 지역 변수를 표시하기 위해 `@` 플래그도 허용합니다.

In [25]:
Cmean = df['C'].mean()
result1 = df[(df.A < Cmean) & (df.B < Cmean)]
result2 = df.query('A < @Cmean and B < @Cmean')
np.allclose(result1, result2)

True

## 성능: 이 기능을 사용해야 하는 경우

`eval`과 `query` 사용 여부를 고려할 때 *계산 시간*과 *메모리 사용*이라는 두 가지 고려 사항이 있습니다.
메모리 사용은 가장 예측 가능한 측면입니다. 이미 언급했듯이 NumPy 배열 또는 Pandas ``DataFrame``과 관련된 모든 복합 표현식은 암시적으로 임시 배열을 생성하게 됩니다. 예를 들면 다음과 같습니다.

In [26]:
x = df[(df.A < 0.5) & (df.B < 0.5)]

대략 다음과 같습니다.

In [27]:
tmp1 = df.A < 0.5
tmp2 = df.B < 0.5
tmp3 = tmp1 & tmp2
x = df[tmp3]

임시 ``DataFrame``의 크기가 사용 가능한 시스템 메모리(일반적으로 수 기가바이트)에 비해 상당한 경우 'eval' 또는 'query' 표현식을 사용하는 것이 좋습니다.
다음을 사용하여 배열의 대략적인 크기를 바이트 단위로 확인할 수 있습니다.

In [28]:
df.values.nbytes

32000

성능 측면에서는 시스템 메모리를 최대로 사용하지 않는 경우에도 `eval`이 더 빠를 수 있습니다.
문제는 임시 개체를 시스템의 L1 또는 L2 CPU 캐시 크기(일반적으로 몇 메가바이트)와 비교하는 방법입니다. 그 값이 훨씬 더 크면 'eval'을 사용하면 서로 다른 메모리 캐시 사이에서 값이 느리게 이동할 가능성이 있는 것을 피할 수 있습니다.
실제로, 전통적인 방법과 `eval`/`query` 방법 사이의 계산 시간 차이는 일반적으로 중요하지 않습니다. 어쨌든 작은 배열의 경우 전통적인 방법이 더 빠릅니다!
`eval`/`query`의 이점은 주로 저장된 메모리와 때로는 제공되는 더 깔끔한 구문에 있습니다.

여기서는 `eval`과 `query`에 대한 대부분의 세부 사항을 다루었습니다. 이에 대한 자세한 내용은 Pandas 설명서를 참조하세요.
특히 이러한 쿼리를 실행하기 위해 다양한 파서와 엔진을 지정할 수 있습니다. 이에 대한 자세한 내용은 문서의 ['성능 향상' 섹션](https://pandas.pydata.org/pandas-docs/dev/user_guide/enhancingperf.html)에 있는 토론을 참조하세요.